# LeetCode 974 — Subarray Sums Divisible by K bằng Python

Link đề: https://leetcode.com/problems/subarray-sums-divisible-by-k/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def subarraysDivByK(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: đáp án là một số nguyên duy nhất. Điểm cốt lõi: tổng đoạn `nums[i..j]` chia hết cho `k` ⇔ hai **tổng tiền tố** `P[i]` và `P[j+1]` có cùng số dư khi chia cho `k`. Bẫy: số dư của số **âm** (Python `%` luôn không âm, C++/Java thì có thể âm).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số đoạn con **không rỗng**, **liên tiếp** có tổng chia hết cho `k`.
2. Nắm công cụ: tổng tiền tố, số dư `% k` với số âm, `dict`/`Counter`/mảng đếm, công thức tổ hợp `c·(c-1)/2`.
3. Đi từ **thử mọi đoạn (`O(n²)`)** → **một lượt với `dict` đếm số dư tiền tố** → **mảng đếm kích thước `k`** → **đếm cặp bằng `Counter` hai lượt** → **chia để trị (đếm cặp qua điểm giữa)**.
4. Hiểu mấu chốt: `sum(nums[i..j]) = P[j+1] - P[i]` chia hết cho `k` ⇔ `P[j+1] ≡ P[i] (mod k)`; mỗi cặp chỉ số tiền tố cùng số dư ứng với đúng một đoạn con.
5. Tránh lỗi kinh điển: quên `count[0] = 1` (tiền tố rỗng), số dư âm (dùng `%` của Python hoặc `((s % k) + k) % k`), đếm theo tổng thay vì theo số dư, công thức tổ hợp sai (`c(c+1)/2`), cập nhật bộ đếm **trước** khi truy vấn.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Trả về **số đoạn con không rỗng** (liên tiếp) có tổng chia hết cho `k`.

**Ví dụ 1:** `nums = [4,5,0,-2,-3,1]`, `k = 5` → `7` (các đoạn: `[4,5,0,-2,-3,1]`, `[5]`, `[5,0]`, `[5,0,-2,-3]`, `[0]`, `[0,-2,-3]`, `[-2,-3]`).

**Ví dụ 2:** `nums = [5]`, `k = 9` → `0`.

**Ràng buộc:** `1 <= nums.length <= 3·10^4`; `-10^4 <= nums[i] <= 10^4`; `2 <= k <= 10^4`.

In [1]:
import itertools
import random
import time
from collections import Counter, defaultdict
from typing import List

examples = [(([4, 5, 0, -2, -3, 1], 5), 7), (([5], 9), 0)]
for (nums, k), want in examples:
    print(f"nums = {nums}, k = {k} -> {want}")

nums = [4, 5, 0, -2, -3, 1], k = 5 -> 7
nums = [5], k = 9 -> 0


## 3. Kiến thức nền

### 3.1 Tổng tiền tố

`P[0] = 0`, `P[i] = nums[0] + … + nums[i-1]`. Khi đó `sum(nums[i..j]) = P[j+1] - P[i]` (`0 ≤ i ≤ j < n`). Mỗi đoạn con không rỗng ứng với **một cặp** chỉ số `(i, j+1)` với `0 ≤ i < j+1 ≤ n`.

In [2]:
nums = [4, 5, 0, -2, -3, 1]
P = [0]
for x in nums:
    P.append(P[-1] + x)
print("P =", P, "| sum(nums[1..3]) =", sum(nums[1:4]), "= P[4] - P[1] =", P[4] - P[1])

P = [0, 4, 9, 9, 7, 4, 5] | sum(nums[1..3]) = 3 = P[4] - P[1] = 3


### 3.2 Chia hết ⇔ cùng số dư

`(P[j+1] - P[i]) % k == 0 ⇔ P[j+1] % k == P[i] % k`. Với **Python**, `a % k` luôn trong `[0, k)` ngay cả khi `a < 0` (`-1 % 5 == 4`). Ở C++/Java, `-1 % 5 == -1` nên phải chuẩn hoá `((a % k) + k) % k`.

In [3]:
import math
print("Python: -1 % 5 =", -1 % 5, "| -7 % 5 =", -7 % 5, "| fmod(-1, 5) =", math.fmod(-1, 5), "(kiểu C: âm)")
print("số dư tiền tố:", [p % 5 for p in P])

Python: -1 % 5 = 4 | -7 % 5 = 3 | fmod(-1, 5) = -1.0 (kiểu C: âm)
số dư tiền tố: [0, 4, 4, 4, 2, 4, 0]


### 3.3 Đếm cặp

Nếu có `c` chỉ số tiền tố cùng một số dư thì chúng tạo `c·(c−1)/2` cặp (tức `c·(c−1)/2` đoạn con). Tổng đáp án = tổng theo từng số dư.

In [4]:
c = Counter(p % 5 for p in P)
print(dict(c), "->", sum(v * (v - 1) // 2 for v in c.values()))

{0: 2, 4: 4, 2: 1} -> 7


## 4. Ý tưởng nền tảng / trực giác

`nums = [4,5,0,-2,-3,1]`, `k = 5`. Tổng tiền tố: `0, 4, 9, 9, 7, 4, 5`; số dư theo `5`: `0, 4, 4, 4, 2, 4, 0`. Các số dư giống nhau:

- số dư `0`: hai chỉ số (đầu và cuối) ⇒ `1` cặp;
- số dư `4`: bốn chỉ số ⇒ `4·3/2 = 6` cặp;
- số dư `2`: một chỉ số ⇒ `0` cặp.

Tổng `1 + 6 + 0 = 7` — đúng đáp án. Thay vì xét mọi đoạn, ta chỉ cần **đếm số dư của các tổng tiền tố**.

In [5]:
mods = [p % 5 for p in P]
groups = defaultdict(list)
for i, m in enumerate(mods):
    groups[m].append(i)
print(dict(groups))
print("cặp:", [(i, j) for idxs in groups.values() for i, j in itertools.combinations(idxs, 2)])

{0: [0, 6], 4: [1, 2, 3, 5], 2: [4]}
cặp: [(0, 6), (1, 2), (1, 3), (1, 5), (2, 3), (2, 5), (3, 5)]


## 5. Brute force — thử mọi đoạn con

**Ý tưởng:** với mỗi điểm bắt đầu `i`, cộng dần `nums[j]` và đếm khi tổng chia hết cho `k`. `O(n²)` với tổng cộng dồn (nếu cộng lại từ đầu mỗi lần thì `O(n³)` — ta dùng bản này làm **oracle** cho mảng nhỏ ở mục 12).

In [6]:
def sub_div_brute(nums, k):
    n, ans = len(nums), 0
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            if s % k == 0:
                ans += 1
    return ans


def sub_div_cubic(nums, k):
    """Oracle theo đúng định nghĩa: tính lại tổng đoạn bằng sum()."""
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i, n) if sum(nums[i:j + 1]) % k == 0)


for (nums, k), want in examples:
    assert sub_div_brute(nums, k) == sub_div_cubic(nums, k) == want
print([sub_div_brute(n, k) for (n, k), _ in examples])

[7, 0]


In [7]:
# Code nộp LeetCode cho Brute force (O(n^2): quá chậm với n = 3·10^4)
class SolutionBrute:
    def subarraysDivByK(self, nums: List[int], k: int) -> int:
        ans = 0
        for i in range(len(nums)):
            s = 0
            for j in range(i, len(nums)):
                s += nums[j]
                if s % k == 0:
                    ans += 1
        return ans

**Độ phức tạp:** `O(n²)` thời gian, `O(1)` bộ nhớ. Với `n = 3·10^4` là `4.5·10^8` phép — quá chậm.

**Nút thắt:** với mỗi điểm kết thúc `j`, ta duyệt lại mọi điểm bắt đầu `i` để kiểm tra `P[j+1] ≡ P[i]`. Chỉ cần biết **đã có bao nhiêu tiền tố trước đó có cùng số dư** — một phép tra bảng.

## 6. Bước đột phá

Duy trì `count[r]` = số tiền tố (kể cả tiền tố rỗng `P[0] = 0`) đã gặp có số dư `r`. Khi đến tiền tố mới có số dư `r`, thêm `count[r]` vào đáp án rồi tăng `count[r]`. `O(n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** một lượt với `dict`, **Cách 3** mảng đếm kích thước `k`, **Cách 4** đếm cặp bằng `Counter` hai lượt, **Cách 5** chia để trị.

### Cách 2 — Một lượt với `dict` đếm số dư tiền tố (đáp án chính)

**Giải thích từng đoạn:**

1. `count = {0: 1}`: tiền tố rỗng có số dư `0`.
2. Duyệt `x`: cộng vào tổng chạy `s`, `r = s % k` (Python luôn không âm).
3. `ans += count.get(r, 0)`: mỗi tiền tố trước đó cùng số dư ứng với một đoạn con kết thúc tại đây.
4. `count[r] += 1` (**sau** khi truy vấn).

In [8]:
def sub_div_dict(nums, k):
    count = {0: 1}                                 # tiền tố rỗng
    s = ans = 0
    for x in nums:
        s += x
        r = s % k                                  # Python: luôn trong [0, k)
        ans += count.get(r, 0)                     # các tiền tố trước cùng số dư
        count[r] = count.get(r, 0) + 1             # cập nhật SAU khi truy vấn
    return ans


for (nums, k), want in examples:
    assert sub_div_dict(nums, k) == want
print([sub_div_dict(n, k) for (n, k), _ in examples])

[7, 0]


In [9]:
# Code nộp LeetCode cho Cách 2
class SolutionDict:
    def subarraysDivByK(self, nums: List[int], k: int) -> int:
        count = {0: 1}
        s = ans = 0
        for x in nums:
            s += x
            r = s % k
            ans += count.get(r, 0)
            count[r] = count.get(r, 0) + 1
        return ans

**Độ phức tạp:** `O(n)` thời gian, `O(min(n, k))` bộ nhớ.

### Cách 3 — Mảng đếm kích thước `k`

**Ý tưởng:** số dư thuộc `[0, k)` nên dùng mảng `cnt = [0] * k` thay `dict` (nhanh, không băm). Khởi tạo `cnt[0] = 1`. Cộng dồn số dư tiền tố ngay (`r = (r + x) % k`) để các số luôn nhỏ.

In [10]:
def sub_div_array(nums, k):
    cnt = [0] * k
    cnt[0] = 1
    r = ans = 0
    for x in nums:
        r = (r + x) % k                            # giữ số dư trong [0, k) (Python)
        ans += cnt[r]
        cnt[r] += 1
    return ans


for (nums, k), want in examples:
    assert sub_div_array(nums, k) == want
print([sub_div_array(n, k) for (n, k), _ in examples])

[7, 0]


In [11]:
# Code nộp LeetCode cho Cách 3
class SolutionArray:
    def subarraysDivByK(self, nums: List[int], k: int) -> int:
        cnt = [0] * k
        cnt[0] = 1
        r = ans = 0
        for x in nums:
            r = (r + x) % k
            ans += cnt[r]
            cnt[r] += 1
        return ans

**Độ phức tạp:** `O(n + k)` thời gian, `O(k)` bộ nhớ.

### Cách 4 — Đếm cặp bằng `Counter` hai lượt

**Ý tưởng:** lượt 1: dựng mọi số dư tiền tố (kể cả `P[0] = 0`) và đếm bằng `Counter`. Lượt 2: với mỗi số dư có `c` lần xuất hiện, cộng `c·(c−1)/2`.

In [12]:
def sub_div_pairs(nums, k):
    mods = [0]
    for x in nums:
        mods.append((mods[-1] + x) % k)
    return sum(c * (c - 1) // 2 for c in Counter(mods).values())


for (nums, k), want in examples:
    assert sub_div_pairs(nums, k) == want
print([sub_div_pairs(n, k) for (n, k), _ in examples])

[7, 0]


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionPairs:
    def subarraysDivByK(self, nums: List[int], k: int) -> int:
        mods = [0]
        for x in nums:
            mods.append((mods[-1] + x) % k)
        return sum(c * (c - 1) // 2 for c in Counter(mods).values())

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ.

### Cách 5 — Chia để trị (đếm các đoạn đi qua điểm giữa)

**Ý tưởng:** `count(l, r)` = số đoạn con trong `nums[l..r]`. Chia tại `mid`: `count(l, mid) + count(mid+1, r) +` số đoạn **cắt qua** `mid`. Một đoạn cắt qua = hậu tố của nửa trái (kết thúc tại `mid`) ghép tiền tố của nửa phải (bắt đầu tại `mid+1`): tổng chia hết ⇔ `(sufL + preR) % k == 0`. Đếm các số dư hậu tố trái bằng `Counter`, rồi với mỗi số dư tiền tố phải `b` cộng `L[(-b) % k]`.

In [14]:
def sub_div_dc(nums, k):
    def solve(l, r):
        if l == r:
            return 1 if nums[l] % k == 0 else 0
        mid = (l + r) // 2
        left_counter = Counter()
        s = 0
        for i in range(mid, l - 1, -1):            # hậu tố của nửa trái: nums[i..mid]
            s += nums[i]
            left_counter[s % k] += 1
        cross = 0
        s = 0
        for j in range(mid + 1, r + 1):            # tiền tố của nửa phải: nums[mid+1..j]
            s += nums[j]
            cross += left_counter[(-s) % k]        # cần (suffix + prefix) % k == 0
        return solve(l, mid) + solve(mid + 1, r) + cross

    return solve(0, len(nums) - 1) if nums else 0


for (nums, k), want in examples:
    assert sub_div_dc(nums, k) == want
print([sub_div_dc(n, k) for (n, k), _ in examples])

[7, 0]


In [15]:
# Code nộp LeetCode cho Cách 5
class SolutionDC:
    def subarraysDivByK(self, nums: List[int], k: int) -> int:
        def solve(l, r):
            if l == r:
                return 1 if nums[l] % k == 0 else 0
            mid = (l + r) // 2
            left = Counter()
            s = 0
            for i in range(mid, l - 1, -1):
                s += nums[i]
                left[s % k] += 1
            cross, s = 0, 0
            for j in range(mid + 1, r + 1):
                s += nums[j]
                cross += left[(-s) % k]
            return solve(l, mid) + solve(mid + 1, r) + cross

        return solve(0, len(nums) - 1)

**Độ phức tạp:** `T(n) = 2T(n/2) + O(n)` ⇒ `O(n log n)` thời gian; `O(n)` bộ nhớ (Counter) + `O(log n)` ngăn xếp.

## 8. TRACE từng bước

**Trace Cách 2** với `nums = [4,5,0,-2,-3,1]`, `k = 5`:

| `x` | `s` | `r = s % 5` | `count` trước | `ans` thêm | `count` sau |
|---|---|---|---|---|---|
| (khởi tạo) | 0 | 0 | — | — | `{0:1}` |
| 4 | 4 | 4 | `{0:1}` | 0 | `{0:1, 4:1}` |
| 5 | 9 | 4 | `{0:1, 4:1}` | 1 | `{0:1, 4:2}` |
| 0 | 9 | 4 | `{0:1, 4:2}` | 2 | `{0:1, 4:3}` |
| -2 | 7 | 2 | `{0:1, 4:3}` | 0 | `{0:1, 4:3, 2:1}` |
| -3 | 4 | 4 | `{0:1, 4:3, 2:1}` | 3 | `{0:1, 4:4, 2:1}` |
| 1 | 5 | 0 | `{0:1, 4:4, 2:1}` | 1 | `{0:2, 4:4, 2:1}` |

Tổng `0 + 1 + 2 + 0 + 3 + 1 = 7` ✓. Hàm trace in từng bước:

In [16]:
def trace_dict(nums, k):
    print(f"nums = {nums}, k = {k}")
    count = {0: 1}
    s = ans = 0
    for x in nums:
        s += x
        r = s % k
        add = count.get(r, 0)
        before = dict(count)
        ans += add
        count[r] = count.get(r, 0) + 1
        print(f"  x={x:3d} s={s:3d} r={r}: count trước {before} -> thêm {add}, ans={ans}")
    print("=>", ans)
    return ans


assert trace_dict([4, 5, 0, -2, -3, 1], 5) == 7

nums = [4, 5, 0, -2, -3, 1], k = 5
  x=  4 s=  4 r=4: count trước {0: 1} -> thêm 0, ans=0
  x=  5 s=  9 r=4: count trước {0: 1, 4: 1} -> thêm 1, ans=1
  x=  0 s=  9 r=4: count trước {0: 1, 4: 2} -> thêm 2, ans=3
  x= -2 s=  7 r=2: count trước {0: 1, 4: 3} -> thêm 0, ans=3
  x= -3 s=  4 r=4: count trước {0: 1, 4: 3, 2: 1} -> thêm 3, ans=6
  x=  1 s=  5 r=0: count trước {0: 1, 4: 4, 2: 1} -> thêm 1, ans=7
=> 7


**Trace số dư âm:** `nums = [2, -3]`, `k = 3`. Tổng tiền tố `P = 0, 2, -1`. Số dư kiểu Python: `0, 2, 2` ⇒ hai tiền tố cùng số dư `2` ⇒ đoạn `[-3]` (tổng `-3` chia hết cho `3`) ✓, đáp án `1`. Nếu dùng phép dư kiểu C/Java (`-1 % 3 = -1`) ta được `0, 2, -1`: không cặp nào trùng ⇒ đáp án sai `0`. Ta kiểm bằng code:

In [17]:
print("Python:", [p % 3 for p in (0, 2, -1)], "| kiểu C (fmod):", [int(math.fmod(p, 3)) for p in (0, 2, -1)])
assert [p % 3 for p in (0, 2, -1)] == [0, 2, 2] and sub_div_dict([2, -3], 3) == 1 == sub_div_cubic([2, -3], 3)

Python: [0, 2, 2] | kiểu C (fmod): [0, 2, -1]


## 9. Code nộp LeetCode

Chọn **Cách 2** (một lượt, `dict`). Giữ nguyên `class Solution`, `subarraysDivByK(self, nums, k)`:

In [18]:
class Solution:
    def subarraysDivByK(self, nums: List[int], k: int) -> int:
        count = {0: 1}                         # tiền tố rỗng (tổng 0) có số dư 0
        s = ans = 0
        for x in nums:
            s += x
            r = s % k                          # Python: luôn trong [0, k) kể cả khi s < 0
            ans += count.get(r, 0)             # mỗi tiền tố trước đó cùng số dư = 1 đoạn con
            count[r] = count.get(r, 0) + 1     # cập nhật sau khi truy vấn
        return ans


print([Solution().subarraysDivByK(n, k) for (n, k), _ in examples])
assert all(Solution().subarraysDivByK(n, k) == w for (n, k), w in examples)

[7, 0]


## 10. Vì sao đúng

**(a) Song ánh giữa đoạn con và cặp tiền tố.** Đoạn không rỗng `nums[i..j]` (`0 ≤ i ≤ j < n`) ↔ cặp `(a, b) = (i, j+1)` với `0 ≤ a < b ≤ n`, và `sum = P[b] − P[a]`.

**(b) Chia hết ⇔ cùng số dư.** `k | P[b] − P[a] ⇔ P[b] ≡ P[a] (mod k)`. Với số nguyên âm, định nghĩa chuẩn của số dư (thuộc `[0, k)`) vẫn thoả quan hệ đồng dư; Python `%` cho đúng giá trị này.

**(c) Đếm.** Đáp án = số cặp `(a, b)`, `a < b`, `P[a] ≡ P[b]`. Xử lý `b` từ trái sang phải; khi tới `b`, số `a < b` cùng số dư chính là `count[r]` (chỉ gồm các tiền tố đã thêm, tức chỉ số nhỏ hơn). Truy vấn **trước** khi thêm `b` vào đảm bảo `a < b` (không tính cặp `(b, b)`).

**(d) `count[0] = 1`.** `P[0] = 0` là một tiền tố hợp lệ (`a = 0`): nó cho phép đếm các đoạn bắt đầu từ đầu mảng (`P[b] ≡ 0`). Thiếu nó ta bỏ sót mọi đoạn tiền tố.

**(e) Cách 3** chỉ thay `dict` bằng mảng (số dư ∈ `[0, k)`), cùng lập luận. **Cách 4** là phát biểu đếm cặp: với mỗi số dư có `c` chỉ số tiền tố ta có `C(c, 2)` cặp, tổng theo số dư.

**(f) Cách 5.** Mỗi đoạn con hoặc nằm hẳn nửa trái, hẳn nửa phải, hoặc cắt qua `mid` (bắt đầu `≤ mid`, kết thúc `> mid`). Đoạn cắt qua ↔ (hậu tố trái `nums[i..mid]`, tiền tố phải `nums[mid+1..j]`) và tổng là tổng hai phần; chia hết ⇔ `(Ls + Rp) % k = 0 ⇔ Rp % k = (−Ls) % k`. Đếm các `Ls % k` bằng `Counter`, rồi tra `L[(−Rp) % k]`.

**(g) Kiểm chứng bằng máy.** Mọi cách khớp oracle `sub_div_cubic` (dùng `sum()` theo định nghĩa) trên **mọi** mảng độ dài ≤ 5 với phần tử trong `[−3, 3]` và mọi `k ∈ [2, 8]`, khớp `O(n²)` trên mảng ngẫu nhiên tới `n = 1500`, và các cách `O(n)`/`O(n log n)` khớp nhau tới `n = 3·10^4` (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi đoạn | `O(n²)` | `O(1)` | oracle `n ≲ 2000` |
| 2. `dict` một lượt | `O(n)` | `O(min(n,k))` | **đáp án chính** |
| 3. Mảng đếm `k` | `O(n + k)` | `O(k)` | nhanh, không băm |
| 4. `Counter` hai lượt | `O(n)` | `O(n)` | dạng khai báo |
| 5. Chia để trị | `O(n log n)` | `O(n)` | minh hoạ đếm cặp |

Đo thực tế `n = 3·10^4`, `k = 10^4`:

In [19]:
def bench(fn, args, repeat=5):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(*args)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(974)
big = [rng.randint(-10 ** 4, 10 ** 4) for _ in range(30000)]
for name, fn in [("dict", sub_div_dict), ("array", sub_div_array), ("pairs", sub_div_pairs), ("dc", sub_div_dc)]:
    print(f"{name:6s}: {bench(fn, (big, 10 ** 4)) * 1e3:7.2f} ms (n = 3e4, k = 1e4)")
print(f"brute : {bench(sub_div_brute, (big[:1500], 7), repeat=1) * 1e3:7.1f} ms (n = 1500)")

dict  :    4.12 ms (n = 3e4, k = 1e4)


array :    2.54 ms (n = 3e4, k = 1e4)


pairs :    3.44 ms (n = 3e4, k = 1e4)


dc    :  112.49 ms (n = 3e4, k = 1e4)
brute :    61.2 ms (n = 1500)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi mảng nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [20]:
IMPLS = {
    "dict": sub_div_dict,
    "array": sub_div_array,
    "pairs": sub_div_pairs,
    "dc": sub_div_dc,
    "brute": sub_div_brute,
    "Solution": lambda n, k: Solution().subarraysDivByK(list(n), k),
    "SolutionDict": lambda n, k: SolutionDict().subarraysDivByK(list(n), k),
    "SolutionArray": lambda n, k: SolutionArray().subarraysDivByK(list(n), k),
    "SolutionPairs": lambda n, k: SolutionPairs().subarraysDivByK(list(n), k),
    "SolutionDC": lambda n, k: SolutionDC().subarraysDivByK(list(n), k),
    "SolutionBrute": lambda n, k: SolutionBrute().subarraysDivByK(list(n), k),
}
SLOW = {"brute", "SolutionBrute"}


def check(nums, k, want=None, names=None, slow_limit=2500):
    if want is None:
        want = sub_div_cubic(nums, k) if len(nums) <= 40 else sub_div_brute(nums, k)
    for name in (names or IMPLS):
        if name in SLOW and len(nums) > slow_limit:
            continue
        assert IMPLS[name](nums, k) == want, (name, nums[:10], k, IMPLS[name](nums, k), want)


for (nums, k), want in examples:
    assert sub_div_cubic(nums, k) == want
    check(nums, k, want)
print("ví dụ đề: oracle định nghĩa và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle định nghĩa và 11 cài đặt đúng


### B. Edge case

- `n = 1`: chia hết / không chia hết; phần tử `0`.
- Toàn số `0`: mọi đoạn đều chia hết ⇒ `n(n+1)/2`.
- Toàn bội của `k` (kể cả âm): `n(n+1)/2`.
- Số âm: `[-1, 2, 9]`, `k = 2`, `[-5]`, `k = 5`.
- Tổng toàn mảng chia hết nhưng không có đoạn con nào khác.
- `k` lớn hơn mọi tổng.
- `k = 2` (chẵn lẻ), phần tử cực đại `±10^4`.
- Tiền tố âm nhiều lần.

In [21]:
edge = [
    ([5], 5), ([5], 9), ([0], 7), ([0, 0, 0, 0], 3), ([7, -7, 14, -14, 0], 7), ([-1, 2, 9], 2), ([-5], 5), ([-1], 5),
    ([1, 2, 3], 6), ([1, 2, 3], 100), ([10 ** 4, -10 ** 4, 10 ** 4], 2), ([10 ** 4] * 10, 10 ** 4), ([-10 ** 4] * 10, 10 ** 4),
    ([1, -1, 1, -1], 2), ([2, -2, 2], 4), ([3, 1, 2, 3], 3), ([-3, -1, -2, -3], 3), ([4, 5, 0, -2, -3, 1], 5), ([1] * 12, 4), ([-1] * 12, 4),
]
for nums, k in edge:
    check(nums, k)
n = 50
assert Solution().subarraysDivByK([0] * n, 3) == n * (n + 1) // 2
assert Solution().subarraysDivByK([7, -14, 21, 0, -7] * 10, 7) == 50 * 51 // 2
print(len(edge), "edge case + toàn 0/toàn bội của k: mọi cài đặt đúng")

20 edge case + toàn 0/toàn bội của k: mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [22]:
def bug_no_init(nums, k):                              # quên count[0] = 1
    count = {}
    s = ans = 0
    for x in nums:
        s += x
        r = s % k
        ans += count.get(r, 0)
        count[r] = count.get(r, 0) + 1
    return ans


def bug_c_style_mod(nums, k):                          # số dư kiểu C (âm) không chuẩn hoá
    count = {0: 1}
    s = ans = 0
    for x in nums:
        s += x
        r = int(math.fmod(s, k))
        ans += count.get(r, 0)
        count[r] = count.get(r, 0) + 1
    return ans


def bug_sum_not_mod(nums, k):                          # đếm theo tổng thay vì số dư
    count = {0: 1}
    s = ans = 0
    for x in nums:
        s += x
        ans += count.get(s, 0)
        count[s] = count.get(s, 0) + 1
    return ans


def bug_pairs_formula(nums, k):                        # c(c+1)/2 thay vì c(c-1)/2
    mods = [0]
    for x in nums:
        mods.append((mods[-1] + x) % k)
    return sum(c * (c + 1) // 2 for c in Counter(mods).values())


def bug_update_before_query(nums, k):                  # cập nhật bộ đếm trước khi truy vấn
    count = {0: 1}
    s = ans = 0
    for x in nums:
        s += x
        r = s % k
        count[r] = count.get(r, 0) + 1
        ans += count[r]
    return ans


def bug_single_elements(nums, k):                      # chỉ đếm đoạn có độ dài 1
    return sum(1 for x in nums if x % k == 0)


def bug_negative_floor(nums, k):                       # chuẩn hoá sai: r = s % k nhưng bỏ phần tử âm
    count = {0: 1}
    s = ans = 0
    for x in nums:
        s += abs(x)
        r = s % k
        ans += count.get(r, 0)
        count[r] = count.get(r, 0) + 1
    return ans


BUGS = [bug_no_init, bug_c_style_mod, bug_sum_not_mod, bug_pairs_formula, bug_update_before_query, bug_single_elements, bug_negative_floor]
probe = [([4, 5, 0, -2, -3, 1], 5), ([5], 5), ([-1, 2, 9], 2), ([1, 2, 3], 6), ([-1, -2, 3], 3), ([2, -2, 2], 4), ([1, 1, 1, 1], 2),
         ([-1, 1], 5), ([3, -3, 3], 3), ([7], 9), ([-3, 1, 2], 3)]
for bug in BUGS:
    found = next(((n, k, bug(n, k), sub_div_cubic(n, k)) for n, k in probe if bug(n, k) != sub_div_cubic(n, k)), None)
    assert found, bug.__name__
    print(f"{bug.__name__:24s} sai trên nums={found[0]}, k={found[1]}: ra {found[2]}, đúng {found[3]}")

bug_no_init              sai trên nums=[4, 5, 0, -2, -3, 1], k=5: ra 6, đúng 7
bug_c_style_mod          sai trên nums=[-1, 2, 9], k=2: ra 1, đúng 2
bug_sum_not_mod          sai trên nums=[4, 5, 0, -2, -3, 1], k=5: ra 2, đúng 7
bug_pairs_formula        sai trên nums=[4, 5, 0, -2, -3, 1], k=5: ra 14, đúng 7
bug_update_before_query  sai trên nums=[4, 5, 0, -2, -3, 1], k=5: ra 13, đúng 7
bug_single_elements      sai trên nums=[4, 5, 0, -2, -3, 1], k=5: ra 2, đúng 7
bug_negative_floor       sai trên nums=[-1, 1], k=5: ra 0, đúng 1


### D. Vét cạn mọi mảng nhỏ

Mọi mảng độ dài `1..5` với phần tử trong `[−3, 3]` (`7 + 49 + 343 + 2401 + 16807 = 19 607` mảng) và mọi `k ∈ [2, 8]` (7 giá trị): `137 249` ca. Oracle: `sub_div_cubic` (tính lại tổng bằng `sum()`).

In [23]:
t0 = time.perf_counter()
cnt = 0
fast = [n for n in IMPLS if n not in SLOW]
for n in range(1, 6):
    for arr in itertools.product(range(-3, 4), repeat=n):
        arr = list(arr)
        for k in range(2, 9):
            want = sub_div_cubic(arr, k)
            for name in fast:
                assert IMPLS[name](arr, k) == want, (name, arr, k)
            if n <= 4:
                assert sub_div_brute(arr, k) == want
            cnt += 1
print(cnt, "ca (mọi mảng ≤ 5 phần tử trong [-3,3], k ∈ [2,8]):", len(fast), "cài đặt nhanh khớp oracle;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 19607 * 7

137249 ca (mọi mảng ≤ 5 phần tử trong [-3,3], k ∈ [2,8]): 9 cài đặt nhanh khớp oracle; 3.6 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) Khớp `O(n²)` brute (với `n ≤ 1500`); (2) cộng bội của `k` vào phần tử **không đổi** đáp án; (3) đảo mảng không đổi đáp án; (4) đổi dấu mọi phần tử không đổi đáp án; (5) `0 ≤ đáp án ≤ n(n+1)/2`; (6) `k = 1` cho `n(n+1)/2` (ngoài ràng buộc, kiểm tra tổng quát); (7) ghép hai mảng: đáp án `≥` tổng đáp án từng mảng.

In [24]:
rng = random.Random(20240974)
t0 = time.perf_counter()
for it in range(300):
    n = rng.randint(1, 1500)
    span = rng.choice([3, 20, 10 ** 4])
    k = rng.choice([2, 3, 7, rng.randint(2, 10 ** 4)])
    nums = [rng.randint(-span, span) for _ in range(n)]
    want = sub_div_brute(nums, k)                                         # (1)
    for name in fast:
        assert IMPLS[name](nums, k) == want, (name, it)
    shifted = [x + k * rng.randint(-3, 3) for x in nums]
    assert Solution().subarraysDivByK(shifted, k) == want                 # (2)
    assert Solution().subarraysDivByK(nums[::-1], k) == want              # (3)
    assert Solution().subarraysDivByK([-x for x in nums], k) == want      # (4)
    assert 0 <= want <= n * (n + 1) // 2                                  # (5)
    assert Solution().subarraysDivByK(nums, 1) == n * (n + 1) // 2        # (6)
    m = rng.randint(1, 200)
    other = [rng.randint(-span, span) for _ in range(m)]
    assert Solution().subarraysDivByK(nums + other, k) >= want + Solution().subarraysDivByK(other, k)   # (7)
print("300 ca ngẫu nhiên (n ≤ 1500): 7 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

300 ca ngẫu nhiên (n ≤ 1500): 7 tính chất đúng; 9.9 s


### F. Giới hạn

`n = 3·10^4`, `|nums[i]| ≤ 10^4`, `k` từ `2` đến `10^4`; mảng toàn `0` (đáp án lớn nhất `n(n+1)/2`), toàn một giá trị, xen kẽ `±`, và các cách `O(n)`/`O(n log n)` khớp nhau.

In [25]:
t0 = time.perf_counter()
n = 30000
rng = random.Random(97401)
cases = [
    ([0] * n, 2, n * (n + 1) // 2),
    ([10 ** 4] * n, 10 ** 4, n * (n + 1) // 2),
    ([-10 ** 4] * n, 10 ** 4, n * (n + 1) // 2),
    ([1, -1] * (n // 2), 2, None),
    ([1] * n, 2, None),
    ([rng.randint(-10 ** 4, 10 ** 4) for _ in range(n)], 10 ** 4, None),
    ([rng.randint(-10 ** 4, 10 ** 4) for _ in range(n)], 2, None),
    ([rng.randint(-10 ** 4, 10 ** 4) for _ in range(n)], 9973, None),
]
for nums, k, want in cases:
    results = {name: IMPLS[name](nums, k) for name in fast}
    assert len(set(results.values())) == 1, results
    if want is not None:
        assert results["Solution"] == want
    print(f"  k = {k:5d}: đáp án = {results['Solution']}")
print("n = 3·10^4: mọi cài đặt nhanh khớp nhau;", round(time.perf_counter() - t0, 2), "s")
# với mảng [1]*n, k = 2: tiền tố 0,1,2,... số dư xen kẽ; đáp án = C(⌈(n+1)/2⌉,2) + C(⌊(n+1)/2⌋,2)
a, b = (n + 2) // 2, (n + 1) // 2
assert Solution().subarraysDivByK([1] * n, 2) == a * (a - 1) // 2 + b * (b - 1) // 2

  k =     2: đáp án = 450015000


  k = 10000: đáp án = 450015000


  k = 10000: đáp án = 450015000


  k =     2: đáp án = 225000000


  k =     2: đáp án = 225000000


  k = 10000: đáp án = 44807
  k =     2: đáp án = 225001332


  k =  9973: đáp án = 45049
n = 3·10^4: mọi cài đặt nhanh khớp nhau; 1.58 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên `count[0] = 1`

In [26]:
nums, k = [5], 5
print("bug:", bug_no_init(nums, k), "| đúng:", Solution().subarraysDivByK(nums, k))
assert bug_no_init(nums, k) == 0 and Solution().subarraysDivByK(nums, k) == 1

bug: 0 | đúng: 1


**Vì sao sai:** tiền tố rỗng `P[0] = 0` cũng là một tiền tố; thiếu nó ta bỏ sót mọi đoạn bắt đầu từ đầu mảng (như `[5]`). **Đúng:** `count = {0: 1}`.

### Lỗi 2: số dư âm không chuẩn hoá (kiểu C)

In [27]:
nums, k = [2, -3], 3
print("bug:", bug_c_style_mod(nums, k), "| đúng:", Solution().subarraysDivByK(nums, k))
assert bug_c_style_mod(nums, k) == 0 and Solution().subarraysDivByK(nums, k) == 1

bug: 0 | đúng: 1


**Vì sao sai:** nếu `s % k` có thể âm (C++/Java) thì `-1` và `k - 1` là cùng một lớp đồng dư nhưng bị coi là khác nhau. **Đúng:** `((s % k) + k) % k` (hoặc dùng `%` của Python).

### Lỗi 3: đếm theo tổng thay vì số dư

In [28]:
nums, k = [4, 5, 0, -2, -3, 1], 5
print("bug:", bug_sum_not_mod(nums, k), "| đúng:", Solution().subarraysDivByK(nums, k))
assert bug_sum_not_mod(nums, k) != 7

bug: 2 | đúng: 7


**Vì sao sai:** hai tiền tố có cùng **tổng** thì hiệu bằng 0 (chia hết cho mọi `k`), nhưng chia hết cho `k` chỉ yêu cầu cùng **số dư**. **Đúng:** khoá của bộ đếm là `P % k`.

### Lỗi 4: công thức tổ hợp sai

In [29]:
print("bug:", bug_pairs_formula(nums, k), "| đúng:", Solution().subarraysDivByK(nums, k))
assert bug_pairs_formula(nums, k) != 7

bug: 14 | đúng: 7


**Vì sao sai:** `c` chỉ số cùng số dư cho `C(c, 2) = c(c−1)/2` cặp (chọn 2 chỉ số khác nhau), không phải `c(c+1)/2`.

### Lỗi 5: cập nhật bộ đếm trước khi truy vấn

In [30]:
print("bug:", bug_update_before_query(nums, k), "| đúng:", Solution().subarraysDivByK(nums, k))
assert bug_update_before_query(nums, k) != 7

bug: 13 | đúng: 7


**Vì sao sai:** cộng `count[r]` **sau** khi đã tăng nó sẽ đếm cả cặp `(b, b)` — đoạn rỗng. **Đúng:** truy vấn rồi mới tăng.

### Lỗi 6 (bonus): chỉ đếm đoạn độ dài 1 / bỏ dấu

In [31]:
nums6 = [-1, 1]
print("độ dài 1:", bug_single_elements([4, 5, 0, -2, -3, 1], 5), "(đúng 7) | abs:", bug_negative_floor(nums6, 5), "| đúng:", Solution().subarraysDivByK(nums6, 5))
assert bug_single_elements([4, 5, 0, -2, -3, 1], 5) == 2 and bug_negative_floor(nums6, 5) == 0 and Solution().subarraysDivByK(nums6, 5) == 1

độ dài 1: 2 (đúng 7) | abs: 0 | đúng: 1


**Vì sao sai:** đề đếm **mọi** đoạn liên tiếp, kể cả dài; và dấu của phần tử ảnh hưởng đến tổng.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "đếm đoạn con có tổng chia hết/đồng dư/bằng một giá trị" → **tổng tiền tố + bảng đếm** (số dư hoặc giá trị).

**Mẹo:**

1. `P[b] − P[a]` là tổng đoạn; chia hết ⇔ cùng số dư.
2. Luôn khởi tạo tiền tố rỗng (`count[0] = 1`).
3. Truy vấn trước, cập nhật sau.
4. Trong ngôn ngữ có `%` âm: chuẩn hoá `((x % k) + k) % k`.
5. Có thể dùng mảng đếm kích thước `k` thay `dict`.
6. Oracle: tính lại tổng bằng `sum()` theo định nghĩa trên mảng nhỏ.

**Họ bài cùng pattern:** Subarray Sum Equals K (560), Continuous Subarray Sum (523), Make Sum Divisible by P (1590), Count Number of Nice Subarrays (1248), Binary Subarrays With Sum (930), Subarrays with K Different Integers (992).

## 15. Biến thể

1. **Đoạn dài nhất** có tổng chia hết cho `k`: lưu **lần đầu** gặp mỗi số dư.
2. **Số dư bằng `r` cho trước** (tổng quát hoá): đếm các cặp có `P[b] − P[a] ≡ r`.
3. **Tổng bằng đúng `target`** (không phải chia hết): bảng đếm tổng tiền tố (`560`).
4. **Tồn tại đoạn độ dài ≥ 2** có tổng chia hết `k` (`523`): bài tập 5.

Cài đặt và đối chiếu brute:

In [32]:
def longest_div(nums, k):
    first = {0: -1}                                 # số dư -> chỉ số tiền tố sớm nhất (chỉ số phần tử cuối, -1 là rỗng)
    s = best = 0
    for j, x in enumerate(nums):
        s += x
        r = s % k
        if r in first:
            best = max(best, j - first[r])
        else:
            first[r] = j
    return best


def count_remainder(nums, k, rem):
    """Số đoạn có tổng ≡ rem (mod k)."""
    count = {0: 1}
    s = ans = 0
    for x in nums:
        s += x
        ans += count.get((s - rem) % k, 0)          # cần P[a] ≡ P[b] - rem
        r = s % k
        count[r] = count.get(r, 0) + 1
    return ans


def count_equal_target(nums, target):
    count = {0: 1}
    s = ans = 0
    for x in nums:
        s += x
        ans += count.get(s - target, 0)
        count[s] = count.get(s, 0) + 1
    return ans


rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 25)
    nums = [rng.randint(-6, 6) for _ in range(n)]
    k = rng.randint(2, 9)
    rem = rng.randint(0, k - 1)
    t = rng.randint(-8, 8)
    segs = [(i, j) for i in range(n) for j in range(i, n)]
    assert longest_div(nums, k) == max([j - i + 1 for i, j in segs if sum(nums[i:j + 1]) % k == 0], default=0)
    assert count_remainder(nums, k, rem) == sum(1 for i, j in segs if sum(nums[i:j + 1]) % k == rem)
    assert count_equal_target(nums, t) == sum(1 for i, j in segs if sum(nums[i:j + 1]) == t)
    assert count_remainder(nums, k, 0) == sub_div_cubic(nums, k)
print("đoạn dài nhất, số dư tuỳ ý, tổng bằng target: khớp vét cạn")

đoạn dài nhất, số dư tuỳ ý, tổng bằng target: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi đoạn | cộng dồn từng `i` | `O(n²)` | `O(1)` | oracle `n ≲ 2000` |
| 2. `dict` một lượt | đếm số dư tiền tố | `O(n)` | `O(min(n,k))` | **đáp án chính** |
| 3. Mảng đếm `k` | như trên, mảng | `O(n+k)` | `O(k)` | `k` nhỏ, nhanh |
| 4. `Counter` hai lượt | `Σ c(c−1)/2` | `O(n)` | `O(n)` | dạng khai báo |
| 5. Chia để trị | cắt qua điểm giữa | `O(n log n)` | `O(n)` | tổng quát hoá khi đếm cặp phức tạp hơn |

Mọi cách khớp oracle định nghĩa trên toàn bộ mảng nhỏ (mục 12D) và khớp nhau tới `n = 3·10^4`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [33]:
print(Solution().subarraysDivByK([4, 5, 0, -2, -3, 1], 5), Solution().subarraysDivByK([5], 9), Solution().subarraysDivByK([0, 0, 0], 4),
      Solution().subarraysDivByK([-1, 2, 9], 2))

7 0 6 2


### Bài 2 (tính tay)

`nums = [2, -2, 3, 0, 4, -7]`, `k = 5`. (a) Viết tổng tiền tố `P` và `P % 5`. (b) Đếm cặp theo số dư. (c) Đáp án? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện cách đếm cặp bằng `Counter`:

In [34]:
def pairs_todo(nums, k):
    mods = [0]
    for x in nums:
        pass            # TODO 1: thêm (mods[-1] + x) % k vào mods
    return 0            # TODO 2: tổng c * (c - 1) // 2 theo từng giá trị trong Counter(mods)


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(pairs_todo(n, k) == w for (n, k), w in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **mảng đếm kích thước `k`**:

In [35]:
class MySolution:
    def subarraysDivByK(self, nums, k):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().subarraysDivByK(n, k) == w for (n, k), w in examples)

### Bài 5 (mini challenge): Continuous Subarray Sum (523)

Viết `has_div_subarray_len2(nums, k)` trả `True` nếu tồn tại đoạn **độ dài ≥ 2** có tổng chia hết cho `k` (`k ≥ 1`). Gợi ý: lưu chỉ số **sớm nhất** của mỗi số dư và kiểm tra khoảng cách `≥ 2`. Kiểm bằng brute `O(n²)`.

In [36]:
def has_div_subarray_len2(nums, k):
    pass            # TODO


# Khi làm xong:
# assert has_div_subarray_len2([23, 2, 4, 6, 7], 6) and not has_div_subarray_len2([23, 2, 6, 4, 7], 13)

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: tổng tiền tố rồi đếm số dư trùng.
- Mức 2: `[0,0,0]`, `k = 4`: mọi đoạn có tổng `0` ⇒ `3 + 2 + 1`.
- Mức 3: kết quả `7 0 6 2` (với `[-1,2,9]`, `k = 2`: hai đoạn `[2]` và `[-1,2,9]` có tổng `2` và `10`).

**Bài 2**

- Mức 1: `P = 0, 2, 0, 3, 3, 7, 0`.
- Mức 2: `P % 5 = 0, 2, 0, 3, 3, 2, 0`.
- Mức 3: số dư `0`: `3` chỉ số ⇒ `3`; số dư `2`: `2` chỉ số ⇒ `1`; số dư `3`: `2` chỉ số ⇒ `1`; đáp án `5`.

**Bài 3**

- Mức 1: `mods.append((mods[-1] + x) % k)`.
- Mức 2: `Counter(mods).values()`.
- Mức 3: `sum(c * (c - 1) // 2 for c in Counter(mods).values())`.

**Bài 4**

- Mức 1: `cnt = [0] * k; cnt[0] = 1`.
- Mức 2: `r = (r + x) % k; ans += cnt[r]; cnt[r] += 1`.
- Mức 3: trả `ans`.

**Bài 5**

- Mức 1: lưu chỉ số tiền tố sớm nhất theo từng số dư (`{0: -1}`).
- Mức 2: gặp lại số dư tại `j` ⇒ độ dài `j - first[r]`; cần `≥ 2`.
- Mức 3: nếu `r` chưa có thì `first[r] = j`; ngược lại nếu `j - first[r] ≥ 2` trả `True`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [37]:
vals = (Solution().subarraysDivByK([4, 5, 0, -2, -3, 1], 5), Solution().subarraysDivByK([5], 9), Solution().subarraysDivByK([0, 0, 0], 4),
        Solution().subarraysDivByK([-1, 2, 9], 2))
print(vals)
assert vals == (7, 0, 6, 2)
print("Bài 1 OK")

(7, 0, 6, 2)
Bài 1 OK


### Bài 2

In [38]:
nums, k = [2, -2, 3, 0, 4, -7], 5
P = [0]
for x in nums:
    P.append(P[-1] + x)
mods = [p % 5 for p in P]
print(P, mods, Counter(mods))
assert P == [0, 2, 0, 3, 3, 7, 0] and mods == [0, 2, 0, 3, 3, 2, 0]
assert Solution().subarraysDivByK(nums, k) == sub_div_cubic(nums, k) == sum(c * (c - 1) // 2 for c in Counter(mods).values()) == 5
print("Bài 2 OK: 5")

[0, 2, 0, 3, 3, 7, 0] [0, 2, 0, 3, 3, 2, 0] Counter({0: 3, 2: 2, 3: 2})
Bài 2 OK: 5


### Bài 3

In [39]:
def pairs_answer(nums, k):
    mods = [0]
    for x in nums:
        mods.append((mods[-1] + x) % k)
    return sum(c * (c - 1) // 2 for c in Counter(mods).values())


assert all(pairs_answer(n, k) == w for (n, k), w in examples)
rng = random.Random(3)
for _ in range(200):
    nums = [rng.randint(-9, 9) for _ in range(rng.randint(1, 30))]
    k = rng.randint(2, 9)
    assert pairs_answer(nums, k) == sub_div_cubic(nums, k)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [40]:
class MySolutionAnswer:
    def subarraysDivByK(self, nums, k):
        cnt = [0] * k
        cnt[0] = 1
        r = ans = 0
        for x in nums:
            r = (r + x) % k
            ans += cnt[r]
            cnt[r] += 1
        return ans


assert all(MySolutionAnswer().subarraysDivByK(n, k) == w for (n, k), w in examples)
rng = random.Random(4)
for _ in range(200):
    nums = [rng.randint(-9, 9) for _ in range(rng.randint(1, 30))]
    k = rng.randint(2, 9)
    assert MySolutionAnswer().subarraysDivByK(nums, k) == sub_div_cubic(nums, k)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [41]:
def has_div_subarray_len2_answer(nums, k):
    first = {0: -1}
    s = 0
    for j, x in enumerate(nums):
        s += x
        r = s % k
        if r not in first:
            first[r] = j
        elif j - first[r] >= 2:
            return True
    return False


def has_brute(nums, k):
    n = len(nums)
    return any(sum(nums[i:j + 1]) % k == 0 for i in range(n) for j in range(i + 1, n))


assert has_div_subarray_len2_answer([23, 2, 4, 6, 7], 6) and not has_div_subarray_len2_answer([23, 2, 6, 4, 7], 13)
rng = random.Random(5)
for _ in range(500):
    nums = [rng.randint(0, 12) for _ in range(rng.randint(1, 12))]
    k = rng.randint(1, 10)
    assert has_div_subarray_len2_answer(nums, k) == has_brute(nums, k), (nums, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (đếm đoạn con có tổng chia hết / đồng dư → tổng tiền tố + bảng đếm số dư):**

```python
class Solution:
    def subarraysDivByK(self, nums: List[int], k: int) -> int:
        count = {0: 1}
        s = ans = 0
        for x in nums:
            s += x
            r = s % k
            ans += count.get(r, 0)
            count[r] = count.get(r, 0) + 1
        return ans
```

**Checklist trước khi submit:**

- [ ] Khởi tạo `count[0] = 1` (tiền tố rỗng)?
- [ ] Khoá là **số dư** `s % k` (không phải tổng)?
- [ ] Số dư không âm (Python `%` ổn; C++/Java chuẩn hoá `((s % k) + k) % k`)?
- [ ] Truy vấn trước, cập nhật sau?
- [ ] Đã thử `[4,5,0,-2,-3,1], 5 → 7`, `[5], 9 → 0`, số âm, toàn `0`?
- [ ] Biết đáp án tối đa `n(n+1)/2` (không tràn với `int` Python)?
- [ ] Đúng tên `class Solution`, `subarraysDivByK(self, nums, k)`, không in debug.